In [1]:
!pip -q install -U sentence-transformers

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path
import hashlib
import json
import os
import time
from datetime import datetime, timezone

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# Google Drive layout
DRIVE_ROOT = Path("/content/drive/MyDrive/MA_Regulatory_Assistant")
INPUT_PATH = DRIVE_ROOT / "colab_input_ocr" / "all_chunks.jsonl"
OUTPUT_DIR = DRIVE_ROOT / "embeddings" / "bge_m3_ocr_v1"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

MODEL_NAME = "BAAI/bge-m3"
EXPECTED_DIMENSION = 1024
MAX_SEQ_LENGTH = 2048
SHARD_SIZE = 256          # nombre de chunks sauvegardés par checkpoint
INITIAL_GPU_BATCH_SIZE = 8
NORMALIZE_EMBEDDINGS = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        f"Fichier introuvable : {INPUT_PATH}\n"
        "Copier all_chunks.jsonl dans MyDrive/MA_Regulatory_Assistant/colab_input_ocr/."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "Aucun GPU CUDA n'est disponible. Dans Colab, choisir "
        "Runtime > Change runtime type > T4 GPU, puis relancer."
    )

GPU_NAME = torch.cuda.get_device_name(0)
print("GPU détecté :", GPU_NAME)
print("CUDA :", torch.version.cuda)
print("Entrée :", INPUT_PATH)
print("Sortie :", OUTPUT_DIR)

GPU détecté : Tesla T4
CUDA : 13.0
Entrée : /content/drive/MyDrive/MA_Regulatory_Assistant/colab_input_ocr/all_chunks.jsonl
Sortie : /content/drive/MyDrive/MA_Regulatory_Assistant/embeddings/bge_m3_ocr_v1


In [4]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def count_jsonl_rows(path: Path) -> int:
    with path.open("r", encoding="utf-8") as stream:
        return sum(1 for line in stream if line.strip())


def iter_jsonl_batches(path: Path, batch_size: int):
    batch = []
    with path.open("r", encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSONL invalide ligne {line_number}: {exc}") from exc

            if not row.get("chunk_id") or not str(row.get("text", "")).strip():
                raise ValueError(
                    f"chunk_id ou text manquant à la ligne {line_number}"
                )

            batch.append(row)
            if len(batch) == batch_size:
                yield batch
                batch = []

    if batch:
        yield batch


def write_jsonl(path: Path, rows):
    with path.open("w", encoding="utf-8", newline="\n") as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False) + "\n")


INPUT_SHA256 = sha256_file(INPUT_PATH)
TOTAL_CHUNKS = count_jsonl_rows(INPUT_PATH)

if TOTAL_CHUNKS == 0:
    raise RuntimeError("Le fichier d'entrée ne contient aucun chunk.")

print("Nombre de chunks :", TOTAL_CHUNKS)
print("SHA-256 entrée :", INPUT_SHA256)

Nombre de chunks : 981
SHA-256 entrée : 1f546758bee4ff4482870a0055205be7e999a0162aeb88f7ed4e0c6badbcb1b7


In [5]:
# Charger le modèle directement sur le GPU Colab.
model = SentenceTransformer(MODEL_NAME, device="cuda")
model.max_seq_length = MAX_SEQ_LENGTH

actual_dimension = model.get_sentence_embedding_dimension()
if actual_dimension != EXPECTED_DIMENSION:
    raise RuntimeError(
        f"Dimension inattendue : {actual_dimension}, attendu : {EXPECTED_DIMENSION}"
    )

print("Modèle :", MODEL_NAME)
print("Dimension :", actual_dimension)
print("Longueur maximale :", model.max_seq_length)
print("GPU :", GPU_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (9 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Modèle : BAAI/bge-m3
Dimension : 1024
Longueur maximale : 2048
GPU : Tesla T4


/tmp/ipykernel_885/1204109349.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  actual_dimension = model.get_sentence_embedding_dimension()


In [6]:
# Vérifier la configuration du checkpoint pour éviter de mélanger deux corpus/modèles.
RUN_CONFIG_PATH = OUTPUT_DIR / "run_config.json"
run_config = {
    "model_name": MODEL_NAME,
    "input_sha256": INPUT_SHA256,
    "total_chunks": TOTAL_CHUNKS,
    "dimension": EXPECTED_DIMENSION,
    "max_seq_length": MAX_SEQ_LENGTH,
    "normalize_embeddings": NORMALIZE_EMBEDDINGS,
    "shard_size": SHARD_SIZE,
}

if RUN_CONFIG_PATH.exists():
    previous = json.loads(RUN_CONFIG_PATH.read_text(encoding="utf-8"))
    if previous != run_config:
        raise RuntimeError(
            "La configuration ou le fichier d'entrée diffère de celle des checkpoints. "
            "Ne pas mélanger les runs. Utiliser un nouveau dossier de sortie "
            "ou archiver l'ancien dossier bge_m3_v1."
        )
else:
    RUN_CONFIG_PATH.write_text(
        json.dumps(run_config, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

print("Configuration de reprise validée.")

Configuration de reprise validée.


In [7]:
def save_numpy_atomic(path: Path, array: np.ndarray):
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as stream:
        np.save(stream, array)
    temporary.replace(path)


def encode_with_oom_retry(texts, initial_batch_size: int):
    batch_size = initial_batch_size
    while True:
        try:
            vectors = model.encode(
                texts,
                batch_size=batch_size,
                normalize_embeddings=NORMALIZE_EMBEDDINGS,
                convert_to_numpy=True,
                show_progress_bar=False,
            )
            vectors = np.asarray(vectors, dtype=np.float32)
            return vectors, batch_size
        except RuntimeError as exc:
            message = str(exc).lower()
            if ("out of memory" not in message) or batch_size <= 1:
                raise
            torch.cuda.empty_cache()
            batch_size = max(1, batch_size // 2)
            print("Mémoire GPU insuffisante : nouveau batch_size =", batch_size)


def shard_paths(shard_index: int):
    stem = f"shard_{shard_index:06d}"
    return (
        CHECKPOINT_DIR / f"{stem}.npy",
        CHECKPOINT_DIR / f"{stem}_ids.jsonl",
    )


def valid_existing_shard(shard_index, rows):
    vector_path, ids_path = shard_paths(shard_index)
    if not vector_path.exists() or not ids_path.exists():
        return False

    try:
        vectors = np.load(vector_path, mmap_mode="r")
        if vectors.shape != (len(rows), EXPECTED_DIMENSION):
            return False

        with ids_path.open("r", encoding="utf-8") as stream:
            ids = [json.loads(line) for line in stream if line.strip()]

        expected_ids = [str(row["chunk_id"]) for row in rows]
        actual_ids = [str(item["chunk_id"]) for item in ids]
        return actual_ids == expected_ids and len(ids) == len(rows)
    except Exception:
        return False


started = time.time()
processed_rows = 0
created_shards = 0
skipped_shards = 0
used_batch_sizes = []

for shard_index, rows in enumerate(iter_jsonl_batches(INPUT_PATH, SHARD_SIZE)):
    if valid_existing_shard(shard_index, rows):
        skipped_shards += 1
        processed_rows += len(rows)
        print(
            f"Checkpoint {shard_index + 1} déjà valide : "
            f"{processed_rows}/{TOTAL_CHUNKS}"
        )
        continue

    texts = [str(row["text"]) for row in rows]

    # Refuser les chunks trop longs plutôt que les tronquer silencieusement.
    token_lengths = [
        len(model.tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])
        for text in texts
    ]
    too_long = [
        {
            "chunk_id": str(rows[i]["chunk_id"]),
            "token_count": int(token_lengths[i]),
        }
        for i in range(len(rows))
        if token_lengths[i] > MAX_SEQ_LENGTH
    ]
    if too_long:
        report_path = OUTPUT_DIR / "chunks_over_max_length.jsonl"
        write_jsonl(report_path, too_long)
        raise RuntimeError(
            f"{len(too_long)} chunk(s) dépassent {MAX_SEQ_LENGTH} tokens. "
            f"Rapport : {report_path}. Corriger le chunking ou augmenter "
            "MAX_SEQ_LENGTH avant de relancer."
        )

    vectors, used_batch_size = encode_with_oom_retry(
        texts, INITIAL_GPU_BATCH_SIZE
    )

    if vectors.shape != (len(rows), EXPECTED_DIMENSION):
        raise RuntimeError(f"Forme de vecteurs incorrecte : {vectors.shape}")

    vector_path, ids_path = shard_paths(shard_index)
    save_numpy_atomic(vector_path, vectors)

    start_row = shard_index * SHARD_SIZE
    id_rows = [
        {
            "row_index": start_row + i,
            "chunk_id": str(row["chunk_id"]),
            "document_id": str(row.get("document_id", "")),
        }
        for i, row in enumerate(rows)
    ]
    temporary_ids = ids_path.with_name(ids_path.name + ".tmp")
    write_jsonl(temporary_ids, id_rows)
    temporary_ids.replace(ids_path)

    created_shards += 1
    processed_rows += len(rows)
    used_batch_sizes.append(used_batch_size)

    print(
        f"Checkpoint {shard_index + 1} terminé : "
        f"{processed_rows}/{TOTAL_CHUNKS} chunks | "
        f"batch GPU={used_batch_size}"
    )

print(f"Calcul terminé en {(time.time() - started) / 60:.1f} minutes.")
print("Nouveaux checkpoints :", created_shards)
print("Checkpoints repris :", skipped_shards)

Checkpoint 1 terminé : 256/981 chunks | batch GPU=8
Checkpoint 2 terminé : 512/981 chunks | batch GPU=8
Checkpoint 3 terminé : 768/981 chunks | batch GPU=8
Checkpoint 4 terminé : 981/981 chunks | batch GPU=8
Calcul terminé en 0.7 minutes.
Nouveaux checkpoints : 4
Checkpoints repris : 0


In [8]:
# Consolider les checkpoints dans une matrice finale et un index d'identifiants.
FINAL_VECTORS = OUTPUT_DIR / "embeddings.npy"
FINAL_IDS = OUTPUT_DIR / "embedding_ids.jsonl"

matrix = np.lib.format.open_memmap(
    FINAL_VECTORS,
    mode="w+",
    dtype=np.float32,
    shape=(TOTAL_CHUNKS, EXPECTED_DIMENSION),
)

id_temp = FINAL_IDS.with_name(FINAL_IDS.name + ".tmp")
expected_row_index = 0

with id_temp.open("w", encoding="utf-8", newline="\n") as ids_out:
    for shard_index, rows in enumerate(iter_jsonl_batches(INPUT_PATH, SHARD_SIZE)):
        vector_path, ids_path = shard_paths(shard_index)

        if not vector_path.exists() or not ids_path.exists():
            raise RuntimeError(f"Checkpoint manquant : shard {shard_index}")

        vectors = np.load(vector_path)
        if vectors.shape != (len(rows), EXPECTED_DIMENSION):
            raise RuntimeError(f"Dimensions invalides pour shard {shard_index}")

        with ids_path.open("r", encoding="utf-8") as ids_in:
            id_rows = [json.loads(line) for line in ids_in if line.strip()]

        if len(id_rows) != len(rows):
            raise RuntimeError(f"Nombre d'identifiants invalide pour shard {shard_index}")

        for local_index, (source_row, id_row) in enumerate(zip(rows, id_rows)):
            if str(source_row["chunk_id"]) != str(id_row["chunk_id"]):
                raise RuntimeError(
                    f"Décalage chunk/vector détecté à la ligne {expected_row_index}"
                )
            if int(id_row["row_index"]) != expected_row_index:
                raise RuntimeError("Ordre row_index incorrect dans les checkpoints")

            matrix[expected_row_index] = vectors[local_index]
            ids_out.write(json.dumps(id_row, ensure_ascii=False) + "\n")
            expected_row_index += 1

matrix.flush()
del matrix
id_temp.replace(FINAL_IDS)

if expected_row_index != TOTAL_CHUNKS:
    raise RuntimeError(
        f"Nombre final incorrect : {expected_row_index} au lieu de {TOTAL_CHUNKS}"
    )

print("Matrice finale :", FINAL_VECTORS)
print("Index des identifiants :", FINAL_IDS)
print("Forme attendue :", (TOTAL_CHUNKS, EXPECTED_DIMENSION))

Matrice finale : /content/drive/MyDrive/MA_Regulatory_Assistant/embeddings/bge_m3_ocr_v1/embeddings.npy
Index des identifiants : /content/drive/MyDrive/MA_Regulatory_Assistant/embeddings/bge_m3_ocr_v1/embedding_ids.jsonl
Forme attendue : (981, 1024)


In [9]:
# Manifeste final avec empreintes et paramètres de reproductibilité.
def sha256_file_final(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


manifest = {
    "status": "completed",
    "model_name": MODEL_NAME,
    "input_file": INPUT_PATH.name,
    "input_sha256": INPUT_SHA256,
    "embedding_file": FINAL_VECTORS.name,
    "embedding_sha256": sha256_file_final(FINAL_VECTORS),
    "ids_file": FINAL_IDS.name,
    "ids_sha256": sha256_file_final(FINAL_IDS),
    "count": TOTAL_CHUNKS,
    "dimension": EXPECTED_DIMENSION,
    "dtype": "float32",
    "normalized": NORMALIZE_EMBEDDINGS,
    "similarity_metric": "cosine",
    "max_seq_length": MAX_SEQ_LENGTH,
    "gpu": GPU_NAME,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "checkpoint_directory": str(CHECKPOINT_DIR),
}

manifest_path = OUTPUT_DIR / "embedding_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n========== LIVRAISON EMBEDDINGS ==========")
print(json.dumps(manifest, ensure_ascii=False, indent=2))


========== LIVRAISON EMBEDDINGS ==========
{
  "status": "completed",
  "model_name": "BAAI/bge-m3",
  "input_file": "all_chunks.jsonl",
  "input_sha256": "1f546758bee4ff4482870a0055205be7e999a0162aeb88f7ed4e0c6badbcb1b7",
  "embedding_file": "embeddings.npy",
  "embedding_sha256": "e5793eff110366c506d260b26b1be001680096c6423064203a8421a5ae4aac56",
  "ids_file": "embedding_ids.jsonl",
  "ids_sha256": "f87b2a1e0123025698ddab525a65cdabf3861dd057eab54778096911b7926160",
  "count": 981,
  "dimension": 1024,
  "dtype": "float32",
  "normalized": true,
  "similarity_metric": "cosine",
  "max_seq_length": 2048,
  "gpu": "Tesla T4",
  "created_at_utc": "2026-11-13T12:48:55.539289+00:00",
  "checkpoint_directory": "/content/drive/MyDrive/MA_Regulatory_Assistant/embeddings/bge_m3_ocr_v1/checkpoints"
}
